# Pipeline Validation

This notebook validates that the modular implementation in `src/`
produces the same cell detections as the original notebook pipeline.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.io.zarr_loader import load_timepoint

from src.preprocessing.pipeline import preprocess_volume
from src.masking.pipeline import create_binary_mask
from src.segmentation.pipeline import segment_instances
from src.detection.pipeline import detect_cells

In [4]:
PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

SAMPLE_PATH = DATA_ROOT / "biohub_5samples_20timepoints" / "train" / f"{SAMPLE_ID}" / f"{SAMPLE_ID}.zarr"

OUTPUT_DIR = (
        DATA_ROOT
        / "processed"
        / "stage_6_processed_dataset"
        / SAMPLE_ID
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [5]:
from tqdm.auto import tqdm
import numpy as np

from src.io.zarr_loader import open_sample, load_timepoint
from src.preprocessing.pipeline import preprocess_volume
from src.masking.pipeline import create_binary_mask
from src.segmentation.pipeline import segment_instances
from src.detection.pipeline import detect_cells

# Open the sample once to determine the number of timepoints
sample = open_sample(SAMPLE_PATH)
num_timepoints = sample.shape[0]

print(f"Processing {num_timepoints} timepoints...")

# Create output directories
preprocessing_dir = OUTPUT_DIR / "preprocessing"
masking_dir = OUTPUT_DIR / "masking"
segmentation_dir = OUTPUT_DIR / "segmentation"
detection_dir = OUTPUT_DIR / "detection"

for directory in (
        preprocessing_dir,
        masking_dir,
        segmentation_dir,
        detection_dir,
):
    directory.mkdir(parents=True, exist_ok=True)

# Process every timepoint
for t in tqdm(range(num_timepoints), desc=SAMPLE_ID):

    volume = load_timepoint(SAMPLE_PATH, t)

    processed = preprocess_volume(volume)

    binary_mask = create_binary_mask(processed)

    instance_labels = segment_instances(binary_mask)

    cells = detect_cells(instance_labels)

    np.save(
        preprocessing_dir / f"t{t:03d}.npy",
        processed,
    )

    np.save(
        masking_dir / f"t{t:03d}.npy",
        binary_mask,
    )

    np.save(
        segmentation_dir / f"t{t:03d}.npy",
        instance_labels,
    )

    cells.to_csv(
        detection_dir / f"t{t:03d}.csv",
        index=False,
    )

print("✅ Processing complete.")

D:\Projects\Kaggle\cell-tracking\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing 20 timepoints...


44b6_0113de3b: 100%|██████████| 20/20 [01:41<00:00,  5.06s/it]

✅ Processing complete.
